<a href="https://colab.research.google.com/github/Valementajat/PageRank_colab/blob/master/pageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [109]:
''' pip install kaggle '''

' pip install kaggle '

In [110]:
''' import os
os.environ['KAGGLE_USERNAME'] = "johannestakamki"
os.environ['KAGGLE_KEY'] = "KGAT_3f821097959746ff4db1bbb5db1fa0f9"
!kaggle datasets download -d  benjaminawd/new-york-times-articles-comments-2020
 '''

' import os\nos.environ[\'KAGGLE_USERNAME\'] = "johannestakamki"\nos.environ[\'KAGGLE_KEY\'] = "KGAT_3f821097959746ff4db1bbb5db1fa0f9"\n!kaggle datasets download -d  benjaminawd/new-york-times-articles-comments-2020\n '

In [111]:
''' # Unzip the file independenty
!unzip new-york-times-articles-comments-2020.zip '''

' # Unzip the file independenty\n!unzip new-york-times-articles-comments-2020.zip '

In [112]:
import numpy as np
import pandas as pd
import random

In [113]:
chunk_size = 10000

ids = []


# To guard against the memory we load the csv in chuncks and process it right away
for article_chunk in pd.read_csv("nyt-articles-2020.csv", chunksize=chunk_size):
   for article_id in article_chunk["uniqueID"]:
      if article_id not in ids:
              ids.append(article_id)
n = len(ids)

In [114]:
# Lets create the dataset we want to work from the articles
sample_frac = 0.2
sample_size = int(n * sample_frac)

sample_ids = random.sample(ids, sample_size)
n = len(sample_ids)
# Maybe this will be a bit more efficient to work with than a full on URL?
id_to_i = {article_id: i for i, article_id in enumerate(sample_ids)}



In [115]:
# Creating an empty adjancency matrix
adjencencyMatrix =  np.zeros((n, n ), dtype=float)



In [116]:
print(ids)

['nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 'nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', 'nyt://article/04bc90f0-b20b-511c-b5bb-3ce13194163f', 'nyt://interactive/5b58d876-9351-50af-9b41-a312490d2728', 'nyt://article/bd8647b3-8ec6-50aa-95cf-2b81ed12d2dd', 'nyt://article/ac742403-9ccd-522f-9a1e-a90feb6c0acd', 'nyt://article/dea10063-7440-586f-bc32-db1dbaa9893f', 'nyt://article/dfd5bb59-f330-5c01-ae02-221e6f9cae7a', 'nyt://article/6227c991-6a3d-5eb1-92c9-a618f6e20dbd', 'nyt://article/7d9a4f78-44ef-5756-bc2b-4c13cbfa0d5a', 'nyt://article/da74e3b3-f027-5b03-9052-0bc2e69a2f3e', 'nyt://article/47b1563f-8bc6-55b3-9db5-74677cc0663c', 'nyt://article/ea265bd2-3c7a-5e5a-a05d-9a924269867b', 'nyt://article/51cb8f6a-ba0b-5af3-aa1d-26e4197b79fe', 'nyt://article/3a42a824-7788-5446-a1fa-500abad98996', 'nyt://article/7fe26134-6bf5-5ca2-b765-932b9deb0ecc', 'nyt://article/3102ac55-dd79-5ad6-b1f5-c2e4bd261e9b', 'nyt://article/8f9752c3-ca7d-55ef-8c58-021e568d92ac', 'nyt://article/c4209597

In [117]:
print(adjencencyMatrix)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [118]:


# drop unnecessary article data from the memory to make space for the comments
# as we don't need them anymore
if 'df_articles' in locals():
    del df_articles

if 'workDataSet' in locals():
    del workDataSet


import gc
gc.collect()

185

In [119]:
KeyValuePairs = []

In [120]:
# TO avoid blowing up the memory usage, we load the
# comments in chucks
chunk_size = 10000
for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size):
  # Start the MapReduce by creating key-value pairs
  for index, row in chunk.iterrows():
    if row['articleID'] in id_to_i:
      KeyValuePairs.append((id_to_i[row["articleID"]], row["userID"]))

del chunk
del chunk_size
import gc
gc.collect()



0

In [121]:
from collections import defaultdict

In [122]:
# Lets group the KeyValuePairs by the key
KeyValueGroup = defaultdict(set)
for article_idx, user_id in KeyValuePairs:
    KeyValueGroup[article_idx].add(user_id)


In [123]:
# Then the final reduce so we can use this
connectionPairs = []

items = list(KeyValueGroup.items())

# This is a possibly a problem?
# I'm checking items against other values in the same set and looking if there
# are two unique intersecting user pairs so we can say that there
for a_idx, users_a in items:
    for b_idx, users_b in items:
        if a_idx != b_idx:
            if len(users_a.intersection(users_b)) >= 2:
                connectionPairs.append((a_idx, b_idx))


In [124]:
for i in range(10):
  print(connectionPairs[i])

(834, 686)
(834, 383)
(834, 2605)
(834, 757)
(834, 2450)
(834, 1639)
(834, 2393)
(834, 2973)
(834, 2128)
(834, 2584)


In [125]:
# degree is practically how many times that page is seen in the connections
degree = Counter()
for i, j in connectionPairs:
    degree[i] += 1


for i, j in connectionPairs:
    adjencencyMatrix[i, j] = 1.0 / degree[i]


In [126]:
#We have a problem, dangling nodes, so this is to uniformally
# set the propability to go from a page to every page when calculating the rank

row_sums = adjencencyMatrix.sum(axis=1)
dangling = (row_sums == 0)


# replace each dangling row with uniform distribution
adjencencyMatrix[dangling, :] = 1.0 / n

In [127]:
# So we have our stochastic adjencencyMatrix or connection matrix

# Without any dangling pages (or nodes)

# Lets run the power iteration

# This is the formula to "teleport", during the power iteration.
# as in, do we take the next page from the page's links or do we take one from
# all of the pages
# P = βM + (1 - β)1/N

beta = 0.85 # example size is between 0.8 - 0.9
n = len(adjencencyMatrix)

#now this is the pageRank transition matrix or the google matrix
P = beta * adjencencyMatrix + (1 - beta) / n

In [128]:
# lets initialize the rank vector r^(0) which has every rank as 1/n
r = np.ones(n) / n

In [129]:
# Now we calculat the new ranks using the transition matrix and the old rank
# r^new =  r^old · A
for _ in range(100):
    r = np.matmul(r, P)

In [130]:
# This is just a sanity check
print(r.sum())

1.000000000000003


In [132]:
# Top 10 indices by rank (largest first)
top_k = 10
top_idx = np.argsort(r)[-top_k:][::-1]

print("Top ranks:")
for idx in top_idx:
    print(idx, r[idx])

Top ranks:
2664 0.0015690888997505793
1492 0.0011919044324000265
2164 0.0011865796455628422
243 0.001162969907020246
2130 0.001143696951085678
3181 0.0011046301430173468
1419 0.0010881770470869147
444 0.0010496945457651237
3084 0.0010484081044554174
222 0.001037856765014155


In [131]:
%whos

Variable           Type           Data/Info
-------------------------------------------
Counter            type           <class 'collections.Counter'>
KeyValueGroup      defaultdict    defaultdict(<class 'set'><...>11771, 34452477, 86527}})
KeyValuePairs      list           n=1015015
P                  ndarray        3357x3357: 11269449 elems, type `float64`, 90155592 bytes (85.97907257080078 Mb)
a_idx              int            448
adjencencyMatrix   ndarray        3357x3357: 11269449 elems, type `float64`, 90155592 bytes (85.97907257080078 Mb)
article_chunk      DataFrame              newsdesk       se<...>n[6787 rows x 11 columns]
article_id         str            nyt://article/12048b2b-62<...>e3-5bed-8c77-483a4299f465
article_idx        int            448
b_idx              int            448
bad                ndarray        232: 232 elems, type `int64`, 1856 bytes
beta               float          0.85
connectionPairs    list           n=2138028
dangling           ndarray      